# 01 - Data Exploration: GI Endoscopy Dataset

This notebook provides an initial exploration of the gastrointestinal endoscopy image dataset (Kvasir / HyperKvasir). We will:

1. **Load the dataset** and verify the directory structure
2. **Visualise class distribution** to check for imbalance
3. **Display sample images** from each disease class
4. **Compute basic statistics** (resolution, aspect ratio, colour channel distributions)

Understanding the data before modelling is essential — class imbalance, resolution variation, and colour biases all influence training and evaluation decisions downstream.

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import src modules
sys.path.insert(0, str(Path.cwd().parent))

sns.set_theme(style="whitegrid", font_scale=1.1)

# ---------- Configuration ----------
DATA_DIR = Path("../data/raw")  # Change this to your dataset path
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif"}

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Exists: {DATA_DIR.exists()}")

## 1. Load Dataset and Discover Classes

We expect a directory-per-class layout where each subdirectory name is a disease class (e.g. `polyps/`, `esophagitis/`). Let's scan the directory and build an inventory.

In [ ]:
# Discover classes and count images per class
class_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
class_names = [d.name for d in class_dirs]

class_counts = {}
all_image_paths = []

for cls_dir in class_dirs:
    paths = sorted([
        p for p in cls_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ])
    class_counts[cls_dir.name] = len(paths)
    all_image_paths.extend([(p, cls_dir.name) for p in paths])

print(f"Found {len(class_names)} classes, {len(all_image_paths)} total images\n")
for name, count in class_counts.items():
    print(f"  {name:30s} {count:>6d} images")

## 2. Class Distribution

A balanced dataset trains more stable classifiers. Severe imbalance may require oversampling, class weighting, or stratified splitting. Let's visualise the distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

names = list(class_counts.keys())
counts = list(class_counts.values())
palette = sns.color_palette("Set2", len(names))

bars = ax.bar(names, counts, color=palette, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Class")
ax.set_ylabel("Number of Images")
ax.set_title("Class Distribution in GI Endoscopy Dataset")
ax.set_xticklabels(names, rotation=45, ha="right")

# Annotate bars with counts
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", va="bottom", fontsize=10)

fig.tight_layout()
plt.show()

# Imbalance ratio
max_count, min_count = max(counts), min(counts)
print(f"\nImbalance ratio (max/min): {max_count / min_count:.2f}x")
print(f"Mean: {np.mean(counts):.0f}, Std: {np.std(counts):.0f}")

## 3. Sample Images from Each Class

Visual inspection helps us understand what the model will see. Notice differences in colour, texture, illumination, and lesion appearance across classes.

In [ ]:
SAMPLES_PER_CLASS = 4

n_classes = len(class_names)
fig, axes = plt.subplots(n_classes, SAMPLES_PER_CLASS, figsize=(4 * SAMPLES_PER_CLASS, 4 * n_classes))

if n_classes == 1:
    axes = [axes]

for row, cls_name in enumerate(class_names):
    cls_dir = DATA_DIR / cls_name
    cls_images = sorted([
        p for p in cls_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ])

    # Pick evenly spaced samples
    indices = np.linspace(0, len(cls_images) - 1, SAMPLES_PER_CLASS, dtype=int)

    for col, idx in enumerate(indices):
        img = cv2.imread(str(cls_images[idx]))
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ax = axes[row][col] if n_classes > 1 else axes[col]
        ax.imshow(rgb)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(cls_name, fontsize=12, rotation=0, labelpad=120, va="center")

fig.suptitle("Sample Images per Class", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()

## 4. Basic Image Statistics

Understanding resolution distribution, aspect ratios, and per-channel intensity statistics helps us choose appropriate preprocessing parameters (resize target, normalisation values).

In [ ]:
# Sample a subset for efficiency (full scan can be slow on large datasets)
rng = np.random.default_rng(42)
sample_size = min(500, len(all_image_paths))
sample_indices = rng.choice(len(all_image_paths), sample_size, replace=False)
sample_paths = [all_image_paths[i] for i in sample_indices]

widths, heights, aspect_ratios = [], [], []
channel_means = {"R": [], "G": [], "B": []}

for img_path, _ in sample_paths:
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]
    widths.append(w)
    heights.append(h)
    aspect_ratios.append(w / h)

    # BGR -> RGB channel means
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    channel_means["R"].append(rgb[:, :, 0].mean())
    channel_means["G"].append(rgb[:, :, 1].mean())
    channel_means["B"].append(rgb[:, :, 2].mean())

print(f"Sampled {sample_size} images for statistics\n")
print(f"Resolution:")
print(f"  Width  — min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}")
print(f"  Height — min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}")
print(f"  Aspect ratio — mean: {np.mean(aspect_ratios):.3f}, std: {np.std(aspect_ratios):.3f}")
print(f"\nChannel means (0-1):")
for ch in ["R", "G", "B"]:
    vals = channel_means[ch]
    print(f"  {ch}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Resolution scatter
axes[0].scatter(widths, heights, alpha=0.4, s=15, edgecolors="none")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")
axes[0].set_title("Image Resolutions")
axes[0].set_aspect("equal")

# Aspect ratio histogram
axes[1].hist(aspect_ratios, bins=30, color=palette[1], edgecolor="black", linewidth=0.5)
axes[1].axvline(np.median(aspect_ratios), color="red", linestyle="--", label=f"Median: {np.median(aspect_ratios):.2f}")
axes[1].set_xlabel("Aspect Ratio (W/H)")
axes[1].set_ylabel("Count")
axes[1].set_title("Aspect Ratio Distribution")
axes[1].legend()

# Per-channel mean distribution
for ch, color in zip(["R", "G", "B"], ["red", "green", "blue"]):
    axes[2].hist(channel_means[ch], bins=30, alpha=0.5, color=color, label=ch, edgecolor="black", linewidth=0.3)
axes[2].set_xlabel("Mean Intensity (0-1)")
axes[2].set_ylabel("Count")
axes[2].set_title("Per-Channel Mean Intensity")
axes[2].legend()

fig.tight_layout()
plt.show()

## 5. Per-Class Intensity Profiles

Different GI conditions may have characteristic colour signatures (e.g. redness in esophagitis, pale mucosa in normal tissue). Let's compare the mean RGB intensity per class.

In [ ]:
# Compute per-class channel means (sample up to 50 images per class)
per_class_rgb = {cls: {"R": [], "G": [], "B": []} for cls in class_names}
SAMPLES_PER = 50

for cls_name in class_names:
    cls_dir = DATA_DIR / cls_name
    cls_images = sorted([
        p for p in cls_dir.iterdir()
        if p.suffix.lower() in IMAGE_EXTENSIONS
    ])
    chosen = rng.choice(len(cls_images), min(SAMPLES_PER, len(cls_images)), replace=False)

    for idx in chosen:
        img = cv2.imread(str(cls_images[idx]))
        if img is None:
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        per_class_rgb[cls_name]["R"].append(rgb[:, :, 0].mean())
        per_class_rgb[cls_name]["G"].append(rgb[:, :, 1].mean())
        per_class_rgb[cls_name]["B"].append(rgb[:, :, 2].mean())

# Grouped bar chart
x = np.arange(len(class_names))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
for i, (ch, color) in enumerate(zip(["R", "G", "B"], ["#e74c3c", "#27ae60", "#2980b9"])):
    means = [np.mean(per_class_rgb[cls][ch]) for cls in class_names]
    stds = [np.std(per_class_rgb[cls][ch]) for cls in class_names]
    ax.bar(x + i * width, means, width, yerr=stds, label=ch, color=color, alpha=0.8,
           capsize=3, edgecolor="black", linewidth=0.3)

ax.set_xticks(x + width)
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_ylabel("Mean Intensity (0-1)")
ax.set_title("Per-Class RGB Channel Means")
ax.legend()
fig.tight_layout()
plt.show()

## Summary

Key observations from this exploration:

- **Class balance**: Check the imbalance ratio above. If >3x, consider weighted sampling or loss weighting during training.
- **Resolution**: Most endoscopy images share a consistent resolution, but resize to 224x224 is still required for ResNet-50.
- **Colour bias**: Endoscopy images are typically red-dominant (mucosal tissue). The red channel mean is usually highest. This is expected and should not be "corrected" during enhancement.
- **Next steps**: Proceed to `02_enhancement_testing.ipynb` to test how enhancement techniques affect these images before training classifiers.